In [1]:
import pandas as pd
import os
import torch
from transformers import AutoTokenizer, AutoModel
import numpy as np

file_path = "../../data/synthetic_report_dataset.csv" # Pastikan nama/path file-nya benar

# 1. Kita intip dulu file mentahnya untuk melihat pemisahnya
with open(file_path, 'r', encoding='utf-8') as f:
    baris_pertama = f.readline()
    print("Isi baris pertama file mentah:\n👉", baris_pertama)

# 2. Auto-Deteksi Pemisah Kolom
if '|' in baris_pertama:
    pemisah = '|'
else:
    pemisah = ','
print(f"Sistem mendeteksi pemisah kolom: '{pemisah}'\n")

# 3. Load Data
nama_kolom = ['teks_keluhan_awam', 'teks_laporan_teknisi', 'kategori_aset', 'severity', 'root_cause', 'tindakan']
df = pd.read_csv(file_path, sep=pemisah, names=nama_kolom, on_bad_lines='skip')

print(f"Baris sebelum dibersihkan: {df.shape[0]}")

# 4. Pembersihan Data Lembut (Soft Cleaning)
# Hanya buang baris kalau KEDUA kolom teksnya kosong (bukan salah satu)
df = df.dropna(subset=['teks_keluhan_awam', 'teks_laporan_teknisi'], how='all')

# Jika baris pertama ternyata adalah header (mengandung kata 'teks_keluhan'), buang baris itu
df = df[df['teks_keluhan_awam'].astype(str).str.contains('teks_keluhan', case=False) == False]

# Reset nomor urut baris
df = df.reset_index(drop=True)

# Pastikan jadi string agar tidak error saat digabung
df['teks_keluhan_awam'] = df['teks_keluhan_awam'].astype(str)
df['teks_laporan_teknisi'] = df['teks_laporan_teknisi'].astype(str)

# 5. Gabungkan Teks (Sebagai Input X)
df['input_teks'] = df['teks_keluhan_awam'] + " " + df['teks_laporan_teknisi']

# 6. MENGAMBIL KOLOM TARGET (Sebagai Kunci Jawaban Y)
kolom_target = ['kategori_aset', 'severity', 'root_cause', 'tindakan']
Y = df[kolom_target]

print(f"Total data SIAP DITRAINING: {df.shape[0]}")
print(f"Bentuk Input (X): 1 Kolom Teks Gabungan")
print(f"Bentuk Target (Y): {Y.shape[1]} Kolom Kunci Jawaban")
print("\n--- Intip Isi Target (Y) ---")
display(Y.head(3)) # Menampilkan 3 baris pertama dari kolom target

d:\05_Personal\College\semester-6\NTG-Project\exigen-smart-maintenance\venv\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


Isi baris pertama file mentah:
👉 AC di ruang meeting lantai 2 netes air terus nih,"Saluran pembuangan air kondensasi tersumbat lumut dan debu, sudah dibersihkan tuntas",HVAC,Sedang,Tersumbat,Pembersihan

Sistem mendeteksi pemisah kolom: ','

Baris sebelum dibersihkan: 666
Total data SIAP DITRAINING: 666
Bentuk Input (X): 1 Kolom Teks Gabungan
Bentuk Target (Y): 4 Kolom Kunci Jawaban

--- Intip Isi Target (Y) ---


,kategori_aset,severity,root_cause,tindakan
0,HVAC,Sedang,Tersumbat,Pembersihan
1,Pompa Air,Tinggi,Keausan,Penggantian Part
2,Mesin Produksi,Tinggi,Tersumbat,Pembersihan


In [2]:
print("Memuat Model IndoBERT... (Tunggu sebentar)")

# Kita gunakan IndoBERT versi dasar yang sangat pintar berbahasa Indonesia
model_name = "indobenchmark/indobert-base-p1"
tokenizer = AutoTokenizer.from_pretrained(model_name)
bert_model = AutoModel.from_pretrained(model_name)

print("IndoBERT berhasil dimuat! Siap mengekstrak makna kalimat.")

Memuat Model IndoBERT... (Tunggu sebentar)


Loading weights: 100%|██████████| 199/199 [00:00<00:00, 20559.30it/s]

IndoBERT berhasil dimuat! Siap mengekstrak makna kalimat.


In [3]:
def get_bert_embedding(text):
    # 1. Tokenisasi teks (Ubah kata jadi ID)
    inputs = tokenizer(text, return_tensors="pt", truncation=True, padding=True, max_length=128)
    
    # 2. Masukkan ke otak IndoBERT (Tanpa update bobot/gradien)
    with torch.no_grad():
        outputs = bert_model(**inputs)
        
    # 3. Ambil vektor [CLS] token yang merepresentasikan makna KESELURUHAN kalimat
    # Vektor ini ukurannya 768 angka desimal
    sentence_embedding = outputs.last_hidden_state[:, 0, :].squeeze().numpy()
    return sentence_embedding

print("Sedang memproses teks menjadi vektor makna (Embeddings)...")
# Ekstrak semua teks kita menjadi list of arrays
embeddings = df['input_teks'].apply(get_bert_embedding)

# Tumpuk menjadi 1 matriks besar (X)
X_deep_learning = np.vstack(embeddings.values)

print("Bentuk Input X sekarang:", X_deep_learning.shape) 
# Outputnya pasti (Jumlah Data, 768). Artinya tiap 1 kalimat diwakili 768 angka bermakna!

Sedang memproses teks menjadi vektor makna (Embeddings)...
Bentuk Input X sekarang: (666, 768)


In [4]:
from sklearn.model_selection import train_test_split
from sklearn.ensemble import RandomForestClassifier
from sklearn.multioutput import MultiOutputClassifier
from sklearn.metrics import accuracy_score

# 1. Bagi data (80% Latih, 20% Uji)
X_train, X_test, y_train, y_test = train_test_split(X_deep_learning, Y, test_size=0.2, random_state=42)

# 2. Inisialisasi Model Multi-Output
rf_classifier = RandomForestClassifier(n_estimators=100, random_state=42)
multi_target_model = MultiOutputClassifier(rf_classifier, n_jobs=-1)

# 3. Proses Latihan (Training)
print("Sedang melatih model...")
multi_target_model.fit(X_train, y_train)

# 4. Evaluasi Akurasi Kejam (Sempurna 4 Kolom/Exact Match)
print("\n--- 1. Akurasi Sempurna (Exact Match Ratio) ---")
print("Catatan: Harus benar tebak 4 kolom sekaligus dalam 1 teks")
train_exact = multi_target_model.score(X_train, y_train)
test_exact = multi_target_model.score(X_test, y_test)
print(f"Akurasi Training: {train_exact * 100:.2f}%")
print(f"Akurasi Testing : {test_exact * 100:.2f}%")

# 5. Evaluasi Akurasi Santai (Per Kolom)
print("\n--- 2. Akurasi Per Target (Individu) ---")
y_train_pred = multi_target_model.predict(X_train)
y_test_pred = multi_target_model.predict(X_test)

# Looping untuk mengecek nilai akurasi di tiap-tiap kolom
for i, kolom in enumerate(kolom_target):
    acc_train = accuracy_score(y_train.iloc[:, i], y_train_pred[:, i])
    acc_test = accuracy_score(y_test.iloc[:, i], y_test_pred[:, i])
    print(f"-> {kolom.upper()}")
    print(f"   Training: {acc_train * 100:.2f}% | Testing: {acc_test * 100:.2f}%")

Sedang melatih model...

--- 1. Akurasi Sempurna (Exact Match Ratio) ---
Catatan: Harus benar tebak 4 kolom sekaligus dalam 1 teks
Akurasi Training: 100.00%
Akurasi Testing : 80.60%

--- 2. Akurasi Per Target (Individu) ---
-> KATEGORI_ASET
   Training: 100.00% | Testing: 96.27%
-> SEVERITY
   Training: 100.00% | Testing: 86.57%
-> ROOT_CAUSE
   Training: 100.00% | Testing: 88.81%
-> TINDAKAN
   Training: 100.00% | Testing: 88.81%
